In [0]:
%sql
-- Intenção: dar a cada perfil o acesso mínimo necessário (princípio do menor privilégio)
GRANT ALL PRIVILEGES ON CATALOG lakehouse_cnes TO `Engenheiros de dados`;

GRANT USE CATALOG ON CATALOG lakehouse_cnes TO `Analistas de dados`;
GRANT USE SCHEMA ON SCHEMA lakehouse_cnes.gold TO `Analistas de dados`;
GRANT SELECT ON SCHEMA lakehouse_cnes.gold TO `Analistas de dados`;

-- Conclusão esperada: analista com leitura apenas na Gold; engenheiro com acesso total ao catálogo
SHOW GRANTS ON SCHEMA lakehouse_cnes.gold;

In [0]:
%sql
-- Intenção: criar a tag governada de classificação de sensibilidade (padrão semáforo)
CREATE GOVERNED TAG classificacao
DESCRIPTION 'Classificação de sensibilidade dos dados: green (público), yellow (interno), red (sensível - LGPD)'
VALUES ('green', 'yellow', 'red');

In [0]:
%sql
-- Intenção: classificar as colunas sensíveis da dimensão publicada na Gold
ALTER TABLE lakehouse_cnes.gold.dim_estabelecimento ALTER COLUMN cpf_cnpj SET TAGS ('classificacao' = 'red');
ALTER TABLE lakehouse_cnes.gold.dim_estabelecimento ALTER COLUMN banco SET TAGS ('classificacao' = 'red');
ALTER TABLE lakehouse_cnes.gold.dim_estabelecimento ALTER COLUMN agencia SET TAGS ('classificacao' = 'red');
ALTER TABLE lakehouse_cnes.gold.dim_estabelecimento ALTER COLUMN conta_corrente SET TAGS ('classificacao' = 'red');
ALTER TABLE lakehouse_cnes.gold.dim_estabelecimento ALTER COLUMN cnpj_mantenedora SET TAGS ('classificacao' = 'yellow');
ALTER TABLE lakehouse_cnes.gold.dim_estabelecimento ALTER COLUMN id_estabelecimento_cnes SET TAGS ('classificacao' = 'green');

In [0]:
%sql
-- Intenção: função de mascaramento aplicada às colunas classificadas como sensíveis
CREATE OR REPLACE FUNCTION lakehouse_cnes.gold.mask_sensivel(valor STRING)
RETURNS STRING
RETURN '***MASCARADO***';

In [0]:
%sql
-- Intenção: política ABAC - mascarar toda coluna 'red' do schema Gold para todos, exceto engenheiros
CREATE POLICY mascara_colunas_sensiveis
ON SCHEMA lakehouse_cnes.gold
COMMENT 'Mascara colunas classificadas como red (CPF e dados bancários) para quem não é engenheiro de dados'
COLUMN MASK lakehouse_cnes.gold.mask_sensivel
TO `account users`
EXCEPT `Engenheiros de dados`
FOR TABLES
MATCH COLUMNS has_tag_value('classificacao', 'red') AS c
ON COLUMN c;

In [0]:
%sql
-- Intenção: view comum (persistente no UC) com a consulta pronta para o analista
CREATE OR REPLACE VIEW lakehouse_cnes.gold.vw_capacidade_competencia_atual
COMMENT 'Capacidade por município na competência mais recente disponível'
AS
SELECT
    f.*
FROM lakehouse_cnes.gold.fato_capacidade_municipio AS f
WHERE f.competencia = (
    SELECT MAX(competencia)
    FROM lakehouse_cnes.gold.fato_capacidade_municipio
);

-- Conclusão esperada: view comum sem armazenamento próprio, recalculada a cada consulta

In [0]:
%sql
-- Intenção: teste do mascaramento (rodar duas vezes: dentro e fora do grupo engenheiro_de_dados)
SELECT
    id_estabelecimento_cnes,
    tipo_pessoa,
    cpf_cnpj,
    banco,
    agencia,
    conta_corrente
FROM lakehouse_cnes.gold.dim_estabelecimento
WHERE banco IS NOT NULL
LIMIT 5;

-- Conclusão esperada: dentro do grupo, valores reais; fora do grupo, '***MASCARADO***' nas colunas red